In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

sns.set_theme(style="whitegrid")

In [2]:
# 1. Load Data - Adjust filename if yours is different
# df = pd.read_csv('telecom_customer_churn.csv').copy()
target_col = 'Churn' 

# Dropping ID column if it exists (e.g., 'customerID')
if 'customerID' in df.columns:
    df = df.drop('customerID', axis=1)

print(f'Shape: {df.shape}')
print('\nFirst 5 rows:')
print(df.head())

print('\nData Types:')
print(df.dtypes)

print('\nMissing Values:')
print(df.isna().sum())

NameError: name 'df' is not defined

In [ ]:
# Target Distribution Plot
plt.figure(figsize=(6,4))
sns.countplot(x=target_col, data=df, palette='coolwarm')
plt.title(f"Distribution of {target_col}")
plt.show()

In [ ]:
# 3. Preprocessing

# Specific fix for common churn dataset issue: TotalCharges as string
if 'TotalCharges' in df.columns:
    df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

# Fill missing values
for col in df.columns:
    if df[col].dtype == 'object':
        df[col] = df[col].fillna(df[col].mode()[0])
    else:
        df[col] = df[col].fillna(df[col].median())

# Encode Categorical features
le = LabelEncoder()
for col in df.select_dtypes(include=['object']).columns:
    df[col] = le.fit_transform(df[col])

X = df.drop(target_col, axis=1)
y = df[target_col]

# Scaling
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# PCA
pca = PCA(n_components=0.95)
X_pca = pca.fit_transform(X_scaled)

# Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X_pca, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Features reduced via PCA: {X_pca.shape[1]}")

In [ ]:
# Using all requested models: Logistic (Lasso), KNN, SVM, Decision Tree, Random Forest
models = [
    ('Logistic_Lasso', GridSearchCV(LogisticRegression(solver='liblinear'), 
                                   {'penalty': ['l1'], 'C': [0.1, 1, 10]}, cv=5)),
    
    ('KNN', GridSearchCV(KNeighborsClassifier(), 
                         {'n_neighbors': [3, 5, 7]}, cv=5)),
    
    ('SVM', GridSearchCV(SVC(), 
                         {'C': [0.1, 1, 10], 'kernel': ['rbf']}, cv=5)),
    
    ('DecisionTree', GridSearchCV(DecisionTreeClassifier(), 
                                  {'max_depth': [None, 5, 10], 'criterion': ['gini', 'entropy']}, cv=5)),
    
    ('RandomForest', GridSearchCV(RandomForestClassifier(), 
                                  {'n_estimators': [50, 100]}, cv=3))
]

results = []

for name, grid in models:
    grid.fit(X_train, y_train)
    best_model = grid.best_estimator_
    y_pred = best_model.predict(X_test)

    results.append({
        'Model': name,
        'Best Params': grid.best_params_,
        'CV Best Score': round(grid.best_score_, 4),
        'Test Accuracy': round(accuracy_score(y_test, y_pred), 4)
    })

# Final Comparison Table
results_df = pd.DataFrame(results).sort_values(by='Test Accuracy', ascending=False).reset_index(drop=True)
print("\nFinal Model Comparison:")
print(results_df)

print(f"\nBest performing model: {results_df.loc[0, 'Model']}")